In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df=pd.read_csv('delhi_10station_pm25_spatiotemporal_dataset_model_ready_v5.csv')
df

,timestamp,station_id,station_name,operating_agency,region_type,zone_description,latitude,longitude,timestamp_ist,temperature_C,...,ist_date,fire_count,fire_frp_sum,fire_count_lag1,fire_frp_lag1,fire_count_lag2,fire_frp_lag2,fire_count_lag3,fire_frp_lag3,urban_category
0,2025-02-18 16:00:00+00:00,10484,aurobindo_marg,DPCC / CPCB,Major Traffic Corridor / Institutional Belt,South Delhi / Sri Aurobindo Marg arterial road...,28.53130,77.200100,2025-02-18 21:30:00+05:30,19.7,...,2025-02-18,22.0,81.53,14.0,33.11,9.0,40.03,17.0,47.54,traffic_corridor
1,2025-02-18 16:00:00+00:00,5570,aya_nagar,IMD / CPCB,Semi-Urban / Border Residential Belt,South Delhi / Ridge & Border Area near Gurugra...,28.47270,77.126500,2025-02-18 21:30:00+05:30,18.0,...,2025-02-18,21.0,78.30,12.0,26.60,9.0,40.03,15.0,30.86,background/rural
2,2025-02-18 16:00:00+00:00,8472,bawana,DPCC / CPCB,Industrial Area / Outer Suburb,North West Delhi / Industrial belt & outer bor...,28.77620,77.051100,2025-02-18 21:30:00+05:30,15.8,...,2025-02-18,23.0,83.30,20.0,43.76,8.0,38.57,18.0,48.78,industrial
3,2025-02-18 16:00:00+00:00,5626,dtu,CPCB,Institutional / Educational Complex Zone,North-West Delhi / Shahbad Daulatpur area; mon...,28.75005,77.111261,2025-02-18 21:30:00+05:30,17.7,...,2025-02-18,23.0,83.30,20.0,49.18,9.0,45.60,18.0,48.78,residential
4,2025-02-18 16:00:00+00:00,5622,nsut,CPCB,Institutional / Suburban Campus Zone,South-West Delhi / Dwarka Sector 3 area; monit...,28.60900,77.032500,2025-02-18 21:30:00+05:30,20.5,...,2025-02-18,20.0,72.99,16.0,36.05,8.0,38.57,17.0,47.54,residential
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116869,2026-08-31 20:00:00+00:00,10484,aurobindo_marg,DPCC / CPCB,Major Traffic Corridor / Institutional Belt,South Delhi / Sri Aurobindo Marg arterial road...,28.53130,77.200100,2026-09-01 01:30:00+05:30,28.5,...,2026-09-01,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,traffic_corridor
116870,2026-08-31 20:00:00+00:00,8472,bawana,DPCC / CPCB,Industrial Area / Outer Suburb,North West Delhi / Industrial belt & outer bor...,28.77620,77.051100,2026-09-01 01:30:00+05:30,28.3,...,2026-09-01,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,industrial
116871,2026-08-31 20:00:00+00:00,50,punjabi_bagh,DPCC / CPCB,Commercial / Residential Transit Zone,West Delhi / Major arterial ring road corridor...,28.67400,77.131000,2026-09-01 01:30:00+05:30,29.5,...,2026-09-01,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,traffic_corridor
116872,2026-08-31 20:00:00+00:00,17,rk_puram,DPCC,Residential / Commercial / High-Density Traffi...,South-West Delhi / Near Sector 8 R.K. Puram an...,28.56420,77.177000,2026-09-01 01:30:00+05:30,28.9,...,2026-09-01,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,residential


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116874 entries, 0 to 116873
Data columns (total 38 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   timestamp                116874 non-null  object 
 1   station_id               116874 non-null  int64  
 2   station_name             116874 non-null  object 
 3   operating_agency         116874 non-null  object 
 4   region_type              116874 non-null  object 
 5   zone_description         116874 non-null  object 
 6   latitude                 116874 non-null  float64
 7   longitude                116874 non-null  float64
 8   timestamp_ist            116874 non-null  object 
 9   temperature_C            116874 non-null  float64
 10  relative_humidity_pct    116874 non-null  int64  
 11  wind_speed_m_s           116874 non-null  float64
 12  wind_direction_deg       116874 non-null  int64  
 13  surface_pressure_hpa     116874 non-null  float64
 14  boun

## IDW

In [4]:
# timestamp is stored as datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])


# Group the data by timestamp
grouped_by_time = dict(tuple(df.groupby('timestamp')))

# Store all predictions
results = []

# Get all station names
all_stations = df['station_name'].unique()


# Hold out one station at a time
for held_out_station in all_stations:

    # Get the held-out station's rows
    station_rows = df[
        df['station_name'] == held_out_station
    ]

    # Get the target station coordinates
    target_lat = station_rows['latitude'].iloc[0]
    target_lon = station_rows['longitude'].iloc[0]

    # Use original PM2.5 as the actual value
    test_rows = station_rows.dropna(
        subset=['pm25_original']
    )


    # Go through each timestamp
    for _, row in test_rows.iterrows():

        # Get timestamp and actual PM2.5
        ts = row['timestamp']
        actual_pm25 = row['pm25_original']

        # Get all stations at this timestamp
        same_time_rows = grouped_by_time.get(ts)

        if same_time_rows is None:
            continue

        # Remove the held-out station
        neighbors = same_time_rows[
            same_time_rows['station_name'] != held_out_station
        ]

        # Keep stations with PM2.5
        neighbors = neighbors.dropna(
            subset=['pm25']
        )

        # Need at least two stations
        if len(neighbors) < 2:
            continue


        # Store the sum of weighted PM2.5
        weighted_pm25_sum = 0

        # Store the sum of weights
        weight_sum = 0


        # Go through every neighbouring station
        for _, neighbor in neighbors.iterrows():

            # Convert coordinates to radians
            lat1 = np.radians(target_lat)
            lon1 = np.radians(target_lon)
            lat2 = np.radians(neighbor['latitude'])
            lon2 = np.radians(neighbor['longitude'])

            # Calculate coordinate differences
            dlat = lat2 - lat1
            dlon = lon2 - lon1

            # Haversine formula
            a = (
                np.sin(dlat / 2) ** 2
                + np.cos(lat1) * np.cos(lat2)
                * np.sin(dlon / 2) ** 2
            )

            # Calculate distance in kilometres
            distance = 2 * 6371 * np.arcsin(np.sqrt(a))

            # Avoid division by zero
            if distance == 0:
                distance = 1e-6

            # IDW weight: 1 / distance²
            weight = 1 / (distance ** 2)

            # Add weighted PM2.5
            weighted_pm25_sum += (
                weight * neighbor['pm25']
            )

            # Add the weight
            weight_sum += weight


        # Final IDW prediction
        predicted_pm25 = (
            weighted_pm25_sum / weight_sum
        )


        # Save the result
        results.append({
            'station_name': held_out_station,
            'timestamp': ts,
            'actual': actual_pm25,
            'predicted': predicted_pm25
        })


# Convert results to dataframe
results_df = pd.DataFrame(results)


# Get actual and predicted values
y_true = results_df['actual']
y_pred = results_df['predicted']

# Calculate overall metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred) ** 0.5
r2 = r2_score(y_true, y_pred)

# Calculate bias
bias = (y_pred - y_true).mean()


print(f"Total predictions made: {len(results_df)}")
print(f"IDW MAE:   {mae:.2f}")
print(f"IDW RMSE:  {rmse:.2f}")
print(f"IDW R²:    {r2:.3f}")
print(f"IDW Bias:  {bias:.2f}")


# Calculate metrics for each held-out station
per_station_error = results_df.groupby(
    'station_name'
).apply(
    lambda g: pd.Series({

        'MAE': mean_absolute_error(
            g['actual'], g['predicted']
        ),

        'RMSE': mean_squared_error(
            g['actual'], g['predicted']
        ) ** 0.5,

        'R2': r2_score(
            g['actual'], g['predicted']
        ),

        'Bias': (
            g['predicted'] - g['actual']
        ).mean(),

        'n_test_points': len(g)
    })
)

print(per_station_error)

Total predictions made: 110158
IDW MAE:   29.43
IDW RMSE:  50.38
IDW R²:    0.636
IDW Bias:  -1.11


C:\Users\dell\AppData\Local\Temp\ipykernel_11864\2424419218.py:149: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


                      MAE       RMSE        R2       Bias  n_test_points
station_name                                                            
anand_vihar     30.073175  47.196743  0.744276 -23.320808        11923.0
aurobindo_marg  25.383198  44.437623  0.436724  22.439763        11901.0
aya_nagar       28.338384  42.170239  0.612569  16.370720         7885.0
bawana          35.663490  58.751829  0.625956 -26.666968        12067.0
dtu             35.604141  62.803467  0.572200  22.047779         9000.0
nsut            40.948678  71.913538  0.035687   9.580049        11777.0
punjabi_bagh    23.748454  37.392584  0.815378 -14.132284        11801.0
pusa            26.907510  39.927255  0.658627  18.802447         9842.0
rk_puram        25.938859  49.816635  0.675935 -21.693310        11945.0
sonia_vihar     22.486905  36.096802  0.810525   0.944694        12017.0


In [5]:
low_pollution = results_df[results_df['actual'] < 100]
high_pollution = results_df[results_df['actual'] >= 100]

print("Low pollution MAE:", (low_pollution['actual'] - low_pollution['predicted']).abs().mean())
print("High pollution MAE:", (high_pollution['actual'] - high_pollution['predicted']).abs().mean())

Low pollution MAE: 19.397060286085516
High pollution MAE: 57.8238905357316


## Nearest Station

In [6]:
# Get all station names
all_stations = df["station_name"].unique()

# Store each station's coordinates
station_info = df[
    ["station_name", "latitude", "longitude"]
].drop_duplicates("station_name").set_index("station_name")

# Group all data by timestamp
timestamp_groups = dict(tuple(df.groupby("timestamp")))

# Store all predictions
all_predictions = []


# Hold out one station at a time
for held_out_station in all_stations:

    print(f"Processing: {held_out_station}")

    # Get the target station's coordinates
    target_lat = station_info.loc[held_out_station, "latitude"]
    target_lon = station_info.loc[held_out_station, "longitude"]

    # Get all data for the target station
    station_rows = df[
        df["station_name"] == held_out_station
    ]

    # Keep only rows with original PM2.5
    station_rows = station_rows.dropna(
        subset=["pm25_original"]
    )


    # Go through each target row
    for row in station_rows.itertuples():

        # Get the timestamp
        timestamp = row.timestamp

        # Get the actual PM2.5
        actual = row.pm25_original

        # Get all stations at this timestamp
        data = timestamp_groups.get(timestamp)

        # Skip if no data exists
        if data is None:
            continue

        # Remove the held-out station
        neighbors = data[
            data["station_name"] != held_out_station
        ]

        # Keep stations with PM2.5
        neighbors = neighbors.dropna(
            subset=["pm25"]
        )

        # Skip if there are no neighbours
        if len(neighbors) == 0:
            continue


        # Start with no nearest station
        nearest_station = None
        nearest_distance = float("inf")
        prediction = None


        # Check every neighbouring station
        for neighbor in neighbors.itertuples():

            # Convert coordinates to radians
            lat1 = np.radians(target_lat)
            lon1 = np.radians(target_lon)
            lat2 = np.radians(neighbor.latitude)
            lon2 = np.radians(neighbor.longitude)

            # Calculate latitude and longitude differences
            dlat = lat2 - lat1
            dlon = lon2 - lon1

            # Haversine formula
            a = (
                np.sin(dlat / 2) ** 2
                + np.cos(lat1) * np.cos(lat2)
                * np.sin(dlon / 2) ** 2
            )

            # Calculate distance in kilometres
            distance = 2 * 6371 * np.arcsin(
                np.sqrt(a)
            )

            # Check if this is the closest station
            if distance < nearest_distance:

                # Save the new shortest distance
                nearest_distance = distance

                # Save the station name
                nearest_station = neighbor.station_name

                # Use its PM2.5 as prediction
                prediction = neighbor.pm25


        # Save the prediction
        all_predictions.append({
            "held_out_station": held_out_station,
            "timestamp": timestamp,
            "nearest_station": nearest_station,
            "nearest_distance_km": nearest_distance,
            "prediction": prediction,
            "actual": actual
        })


# Convert predictions to a dataframe
results = pd.DataFrame(all_predictions)

print("\nTotal predictions:", len(results))


# Get actual and predicted values
y_true = results["actual"]
y_pred = results["prediction"]

# Calculate MAE
mae = mean_absolute_error(y_true, y_pred)

# Calculate RMSE
rmse = mean_squared_error(y_true, y_pred) ** 0.5

# Calculate R²
r2 = r2_score(y_true, y_pred)

# Calculate bias
bias = (y_pred - y_true).mean()


print("\n======================================")
print("NEAREST-STATION BASELINE")
print("======================================")

print(f"MAE  : {mae:.2f} µg/m³")
print(f"RMSE : {rmse:.2f} µg/m³")
print(f"Bias : {bias:.2f} µg/m³")
print(f"R²   : {r2:.3f}")


# Store per-station metrics
station_metrics = []


# Calculate metrics for each held-out station
for station, group in results.groupby("held_out_station"):

    # Calculate MAE
    station_mae = mean_absolute_error(
        group["actual"],
        group["prediction"]
    )

    # Calculate RMSE
    station_rmse = mean_squared_error(
        group["actual"],
        group["prediction"]
    ) ** 0.5

    # Calculate R²
    station_r2 = r2_score(
        group["actual"],
        group["prediction"]
    )

    # Calculate bias
    station_bias = (
        group["prediction"] - group["actual"]
    ).mean()

    # Save the results
    station_metrics.append({
        "station": station,
        "MAE": station_mae,
        "RMSE": station_rmse,
        "R2": station_r2,
        "Bias": station_bias,
        "n_test_points": len(group)
    })


# Convert metrics to a dataframe
station_metrics = pd.DataFrame(station_metrics)


# Keep the same station order
station_order = [
    "anand_vihar",
    "aurobindo_marg",
    "aya_nagar",
    "bawana",
    "dtu",
    "nsut",
    "punjabi_bagh",
    "pusa",
    "rk_puram",
    "sonia_vihar"
]

# Set the station order
station_metrics["station"] = pd.Categorical(
    station_metrics["station"],
    categories=station_order,
    ordered=True
)

# Sort the results
station_metrics = station_metrics.sort_values("station")


print("\n======================================")
print("PER-STATION RESULTS")
print("======================================")

print(station_metrics.to_string(index=False))

Processing: aurobindo_marg
Processing: aya_nagar
Processing: bawana
Processing: dtu
Processing: nsut
Processing: punjabi_bagh
Processing: pusa
Processing: rk_puram
Processing: sonia_vihar
Processing: anand_vihar

Total predictions: 110160

NEAREST-STATION BASELINE
MAE  : 34.25 µg/m³
RMSE : 58.78 µg/m³
Bias : -0.74 µg/m³
R²   : 0.504

PER-STATION RESULTS
       station       MAE      RMSE       R2       Bias  n_test_points
   anand_vihar 32.280904 50.967691 0.701780 -18.414631          11923
aurobindo_marg 30.555830 57.231423 0.065695  27.056571          11901
     aya_nagar 24.364130 38.740874 0.673020   0.671650           7885
        bawana 41.536864 70.856142 0.455955 -29.846411          12067
           dtu 46.995091 79.172068 0.320144  35.613734           9000
          nsut 40.889498 71.976770 0.033990  -1.434954          11777
  punjabi_bagh 31.249667 49.227101 0.680021 -20.622439          11801
          pusa 31.058592 47.538511 0.516023  22.243255           9844
      rk_puram

In [7]:
low = results[results["actual"] < 100]
high = results[results["actual"] >= 100]

print("Nearest - Low pollution MAE:",
      mean_absolute_error(low["actual"], low["prediction"]))

print("Nearest - High pollution MAE:",
      mean_absolute_error(high["actual"], high["prediction"]))

Nearest - Low pollution MAE: 23.153932168341868
Nearest - High pollution MAE: 65.62993530542157


In [8]:
# Convert timestamp values to UTC datetime objects
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

# Sort the data so each station's observations are in chronological order
df = df.sort_values(["station_name", "timestamp"])


# ============================================================
# 1. BASIC TIME COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("1. TIME COVERAGE PER STATION")
print("=" * 70)

# Find the first timestamp, last timestamp, and number of records for each station
coverage = (
    df.groupby("station_name")["timestamp"]
    .agg(["min", "max", "count"])
)

# Calculate how many days of data are covered by each station
coverage["duration_days"] = (
    coverage["max"] - coverage["min"]
).dt.total_seconds() / 86400

print(coverage)


# ============================================================
# 2. TIMESTAMP FREQUENCY
# ============================================================

print("\n" + "=" * 70)
print("2. TIME STEP DISTRIBUTION")
print("=" * 70)

# Check how frequently observations are recorded at each station
for station, group in df.groupby("station_name"):

    # Calculate the time difference between consecutive observations
    diffs = group["timestamp"].diff().dropna()

    print(f"\n{station}")
    print(diffs.describe())

    print("\nMost common intervals:")

    # Show the time intervals that occur most often
    print(
        diffs.value_counts().head(10)
    )


# ============================================================
# 3. LARGE GAPS PER STATION
# ============================================================

print("\n" + "=" * 70)
print("3. LARGE TIMESTAMP GAPS")
print("=" * 70)

# Store information about all gaps longer than one hour
gap_records = []

# Check timestamp gaps separately for each station
for station, group in df.groupby("station_name"):

    # Sort timestamps before calculating consecutive differences
    timestamps = group["timestamp"].sort_values()

    # Calculate the gap between each timestamp and the previous one
    diffs = timestamps.diff()

    # Keep only gaps longer than one hour
    large_gaps = diffs[diffs > pd.Timedelta(hours=1)]

    # Record the start, end, and length of each large gap
    for idx, gap in large_gaps.items():

        gap_records.append({
            "station": station,
            "gap_start": timestamps.loc[idx] - gap,
            "gap_end": timestamps.loc[idx],
            "gap_hours": gap.total_seconds() / 3600
        })

# Convert the recorded gaps into a DataFrame
gaps = pd.DataFrame(gap_records)

if len(gaps) > 0:

    print("\nNumber of gaps > 1 hour:")

    # Count how many large gaps occurred at each station
    print(
        gaps.groupby("station").size()
    )

    print("\nLargest gaps:")

    # Display the 30 longest gaps across all stations
    print(
        gaps.sort_values(
            "gap_hours",
            ascending=False
        ).head(30).to_string(index=False)
    )

else:
    print("No gaps > 1 hour found.")


# ============================================================
# 4. REALLY LARGE GAPS
# ============================================================

print("\n" + "=" * 70)
print("4. GAPS > 24 HOURS")
print("=" * 70)

if len(gaps) > 0:

    # Keep only gaps that last more than one full day
    huge_gaps = gaps[
        gaps["gap_hours"] > 24
    ]

    # Count the number of gaps longer than 24 hours at each station
    print(
        huge_gaps
        .groupby("station")
        .size()
        .sort_values(ascending=False)
    )

    print("\nLargest >24h gaps:")

    # Show the longest gaps that lasted more than 24 hours
    print(
        huge_gaps
        .sort_values(
            "gap_hours",
            ascending=False
        )
        .head(30)
        .to_string(index=False)
    )


# ============================================================
# 5. ORIGINAL PM2.5 AVAILABILITY
# ============================================================

print("\n" + "=" * 70)
print("5. ORIGINAL PM2.5 AVAILABILITY")
print("=" * 70)

# Count the total rows and the number of original PM2.5 measurements
pm25_availability = (
    df.groupby("station_name")
    .agg(
        total=("pm25_original", "size"),
        original_available=(
            "pm25_original",
            "count"
        )
    )
)

# Calculate the percentage of rows with an original PM2.5 measurement
pm25_availability["availability_%"] = (
    pm25_availability["original_available"]
    / pm25_availability["total"]
    * 100
)

print(pm25_availability)


# ============================================================
# 6. HOW MANY STATIONS EXIST AT EACH TIMESTAMP?
# ============================================================

print("\n" + "=" * 70)
print("6. STATION COVERAGE AT EACH TIMESTAMP")
print("=" * 70)

# Count how many different stations have data at each timestamp
timestamp_station_count = (
    df.groupby("timestamp")["station_name"]
    .nunique()
)

# Summarize the number of available stations across timestamps
print(
    timestamp_station_count.describe()
)

print("\nDistribution:")

# Show how often each number of available stations occurs
print(
    timestamp_station_count
    .value_counts()
    .sort_index()
)


# ============================================================
# 7. HOW OFTEN DO WE HAVE >=2, >=3, ... STATIONS?
# ============================================================

print("\n" + "=" * 70)
print("7. SIMULTANEOUS STATION AVAILABILITY")
print("=" * 70)

# Get the total number of unique timestamps
total_timestamps = len(timestamp_station_count)

# Check how often at least 1, 2, 3, ... 10 stations are available
for n in range(1, 11):

    # Count timestamps with at least n available stations
    count = (
        timestamp_station_count >= n
    ).sum()

    # Convert that count into a percentage of all timestamps
    percentage = (
        count / total_timestamps * 100
    )

    print(
        f">= {n} stations: "
        f"{count:,} timestamps "
        f"({percentage:.2f}%)"
    )


# ============================================================
# 8. LOSO NEIGHBOR AVAILABILITY
# ============================================================

print("\n" + "=" * 70)
print("8. LOSO NEIGHBOR AVAILABILITY")
print("=" * 70)

# Check whether enough other stations are available
# when each station is treated as the held-out station
for station in df["station_name"].unique():

    # Get timestamps where the held-out station has an original PM2.5 value
    target = df[
        df["station_name"] == station
    ][["timestamp", "pm25_original"]].dropna()

    # Count how many other stations have PM2.5 data at each timestamp
    available_neighbors = (
        df[
            df["station_name"] != station
        ]
        .groupby("timestamp")["station_name"]
        .nunique()
    )

    # Match the available neighbor count to the target timestamps
    merged = target.merge(
        available_neighbors.rename(
            "available_neighbors"
        ),
        left_on="timestamp",
        right_index=True,
        how="left"
    )

    # Treat timestamps with no available neighbors as zero
    merged["available_neighbors"] = (
        merged["available_neighbors"]
        .fillna(0)
    )

    print(f"\n{station}")

    # Check several useful neighbor thresholds for LOSO validation
    for n in [1, 2, 3, 5, 8, 9]:

        # Calculate the percentage of target timestamps
        # where at least n neighboring stations are available
        pct = (
            merged["available_neighbors"] >= n
        ).mean() * 100

        print(
            f"  >= {n} neighbors: {pct:.2f}%"
        )


# ============================================================
# 9. DUPLICATE TIMESTAMPS
# ============================================================

print("\n" + "=" * 70)
print("9. DUPLICATE STATION + TIMESTAMP")
print("=" * 70)

# Identify rows where the same station has the same timestamp more than once
duplicates = df.duplicated(
    subset=["station_name", "timestamp"],
    keep=False
)

# Count all rows involved in duplicate station-timestamp combinations
print(
    f"Duplicate rows: {duplicates.sum():,}"
)

if duplicates.sum() > 0:

    # Display some duplicate station-timestamp combinations for inspection
    print(
        df.loc[
            duplicates,
            ["station_name", "timestamp"]
        ]
        .sort_values(
            ["station_name", "timestamp"]
        )
        .head(20)
    )


# ============================================================
# 10. FINAL DATASET VERDICT
# ============================================================

print("\n" + "=" * 70)
print("10. SUMMARY")
print("=" * 70)

# Print the final size and basic structure of the dataset
print(f"Rows: {len(df):,}")
print(f"Stations: {df['station_name'].nunique()}")
print(
    f"Unique timestamps: "
    f"{df['timestamp'].nunique():,}"
)


1. TIME COVERAGE PER STATION
                                     min                       max  count  \
station_name                                                                
anand_vihar    2025-02-18 18:00:00+00:00 2026-08-31 08:00:00+00:00  12718   
aurobindo_marg 2025-02-18 16:00:00+00:00 2026-08-31 20:00:00+00:00  12561   
aya_nagar      2025-02-18 16:00:00+00:00 2026-04-14 12:00:00+00:00   8463   
bawana         2025-02-18 16:00:00+00:00 2026-08-31 20:00:00+00:00  12729   
dtu            2025-02-18 16:00:00+00:00 2026-04-03 16:00:00+00:00   9339   
nsut           2025-02-18 16:00:00+00:00 2026-08-30 20:00:00+00:00  12347   
punjabi_bagh   2025-02-18 16:00:00+00:00 2026-08-31 20:00:00+00:00  12610   
pusa           2025-02-18 16:00:00+00:00 2026-07-16 05:00:00+00:00  10814   
rk_puram       2025-02-18 16:00:00+00:00 2026-08-31 20:00:00+00:00  12663   
sonia_vihar    2025-02-18 16:00:00+00:00 2026-08-31 20:00:00+00:00  12630   

                duration_days  
station_name 